# Dental Panoramic AI — full-resolution GPU run (Colab)

**Project:** Tufts + DENTEX cross-dataset panoramic X-ray AI
([boakyejeff/dental-panoramic-ai](https://github.com/boakyejeff/dental-panoramic-ai)).
**What this notebook does:** trains the full-resolution detector the local
CPU box cannot run — **unfrozen ResNet50-FPN at 1024×512** (the CPU run was
MobileNetV3 frozen at 768×384; mAP 0.27, caries AP only 0.12, anterior
teeth 0/14) — plus the mixed-site ResNet18 classifier on GPU, evaluates
both cross-site, and saves everything to Google Drive.

**Before you run anything:**
1. **Runtime > Change runtime type > T4 GPU** (free). If no T4 is
   available, Colab Pro ($10/mo) gives an L4, Pro+ ($50/mo) an A100.
   This notebook does **not** choose the GPU for you — you pick it in
   that menu, then re-run the GPU-check cell to confirm.
2. Sign in to Google when Drive asks, and put the two Tufts zips in
   Drive first (see the Tufts cell below): `Radiographs.zip` (119 MB)
   and `Expert.zip` (49 MB) in `MyDrive/dental-panoramic-ai/`.
3. Run the cells **top to bottom, one at a time**. The detector trains
   ~1–2 h on a T4 (~40–60 min L4, ~20–30 min A100); a checkpoint is
   saved after every epoch, so a disconnect costs at most one epoch.

**What stays where:** nothing is uploaded to GitHub from here. Results
and checkpoints are copied to `MyDrive/dental-panoramic-ai/` at the end;
copy the new `*_gpu.json` numbers back into the manuscript by hand.


In [ ]:
# 1 — Confirm the GPU. If this says "No GPU", stop:
# Runtime > Change runtime type > T4 GPU, then run this cell again.
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2 — Mount Google Drive

This is where the Tufts zips come from and where all results go, so a
Colab disconnect does not lose the run.


In [ ]:
# 2 — Mount Drive (sign in with the Google account that has the Tufts zips).
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")


## 3 — Get the project code

Clones the GitHub repo (public, no login needed). If you edited the
scripts locally since the last push, push first — Colab always uses
what is on GitHub, not your local disk.


In [ ]:
# 3 — Clone the repo (or pull the latest if it is already here).
import os
PROJECT = "/content/dental-panoramic-ai"
if os.path.isdir(PROJECT):
    !cd {PROJECT} && git pull --ff-only
else:
    !git clone https://github.com/boakyejeff/dental-panoramic-ai.git {PROJECT}
%cd {PROJECT}
!ls
assert os.path.isfile("colab/train_detector_gpu.py"), "GPU scripts missing — check the clone"
print("Project ready at", PROJECT)


## 4 — Install dependencies

**Do NOT `pip install -r requirements.txt` here.** That file pins
`torch==2.14.0+cpu`, which would replace Colab's CUDA PyTorch with the
CPU build and silently put training back on the CPU. Colab already
ships CUDA PyTorch — install only the small packages.


In [ ]:
# 4 — Small packages only. Colab's CUDA torch is kept as-is.
!pip install -q numpy pandas scikit-learn matplotlib Pillow
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Still no GPU — set Runtime > Change runtime type > T4 GPU"


## 5 — Download DENTEX (Zenodo, public, no login)

Training archive is 10.9 GB — the download takes ~5–15 min in Colab.
MD5s are checked against the Zenodo record (7812323); a mismatch stops
the notebook, because a corrupt 10.9 GB download must not train silently.


In [ ]:
# 5 — DENTEX from Zenodo (record 7812323), with MD5 verification.
import hashlib, os, subprocess
DATA = "data"
os.makedirs(DATA, exist_ok=True)

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

FILES = [
    ("training_data.zip", "https://zenodo.org/api/records/7812323/files/training_data.zip/content",
     "eba0c44fa4c6e8ec221db3009eccc1f9", 10927391529),
    ("validation_data.zip", "https://zenodo.org/api/records/7812323/files/validation_data.zip/content",
     "b2a13267740185e42cb59e968bc3ce83", 149512256),
]
for name, url, want_md5, want_size in FILES:
    p = os.path.join(DATA, name)
    if os.path.exists(p) and os.path.getsize(p) == want_size and md5(p) == want_md5:
        print(f"{name}: already present and MD5-verified, skipping download")
        continue
    print(f"downloading {name} ...")
    subprocess.run(["curl", "-L", "-o", p, url], check=True)
    assert os.path.getsize(p) == want_size, f"{name}: size mismatch"
    assert md5(p) == want_md5, f"{name}: MD5 mismatch — re-download"
    print(f"{name}: OK ({want_size} bytes, MD5 verified)")

# Unzip (training zip contains training_data/ + the triple JSONs).
subprocess.run(["unzip", "-q", "-o", f"{DATA}/training_data.zip", "-d", f"{DATA}/training"], check=True)
subprocess.run(["unzip", "-q", "-o", f"{DATA}/validation_data.zip", "-d", f"{DATA}/validation_data"], check=True)
# Validation triple JSON (COCO-like; the local scripts read it as validation_triple2.json).
subprocess.run(["curl", "-sL", "-o", f"{DATA}/validation_triple2.json",
                "https://raw.githubusercontent.com/ibrahimethemhamamci/DENTEX/main/DENTEX/validation_triple.json"],
               check=True)
!ls data/training/training_data/ && ls data/validation_data/ | head
print("DENTEX ready.")


## 6 — Tufts data

**There is no scriptable direct download for Tufts.** The Tufts site
(http://tdd.ece.tufts.edu/) is a JavaScript page and its zip links
return 403 to scripts, so this cell does not pretend to download it.
Use one of these instead:

- **Recommended:** upload `Radiographs.zip` (119 MB) and `Expert.zip`
  (49 MB) — the files already on the local machine — to
  `MyDrive/dental-panoramic-ai/` in Google Drive, then run this cell.
- **Alternative:** the Kaggle mirror
  (`deepologylab/tufts-dental-database`) with a Kaggle API token
  (`kaggle.json` in Drive or `~/.kaggle/`). Needs a Kaggle account.


In [ ]:
# 6 — Tufts from Drive (recommended), Kaggle mirror as fallback.
import os, shutil, subprocess, glob

CANDIDATES = [
    "/content/drive/MyDrive/dental-panoramic-ai",
    "/content/drive/MyDrive/dental-panoramic-ai/data",
    "/content/drive/MyDrive",
]
found = {}
for d in CANDIDATES:
    for name in ("Radiographs.zip", "Expert.zip"):
        p = os.path.join(d, name)
        if os.path.isfile(p):
            found.setdefault(name, p)
missing = [n for n in ("Radiographs.zip", "Expert.zip") if n not in found]
if missing:
    # Kaggle fallback (needs ~/.kaggle/kaggle.json or KAGGLE_USERNAME/KEY).
    print(f"Not in Drive: {missing}. Trying Kaggle mirror ...")
    r = subprocess.run(["kaggle", "datasets", "download", "-d",
                        "deepologylab/tufts-dental-database", "-p", "data", "--unzip"],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    if r.returncode != 0:
        raise SystemExit(
            "Tufts zips not found. Upload Radiographs.zip + Expert.zip to "
            "MyDrive/dental-panoramic-ai/ in Google Drive and re-run this cell, "
            "or set up a Kaggle API token for the deepologylab mirror.")
for name, src in found.items():
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        print(f"copying {src} -> {dst}")
        shutil.copy(src, dst)
    print(name, os.path.getsize(dst), "bytes")
subprocess.run(["unzip", "-q", "-o", "data/Radiographs.zip", "-d", "data/Radiographs"], check=True)
subprocess.run(["unzip", "-q", "-o", "data/Expert.zip", "-d", "data/Expert"], check=True)
print("Tufts ready:", len(glob.glob("data/Radiographs/*.JPG")), "radiographs")


## 7 — Build resized image caches

The training scripts read resized JPEG caches, not the raw PNGs/JPGs.
1024×512 is what the GPU detector trains on; 512×256 is what the
classifier uses. ~5 min.


In [ ]:
# 7 — Write the cache builder, then run it.
%%writefile colab/build_caches.py
"""Build 1024x512 + 512x256 JPEG caches (Colab). Skips files already built."""
import json
from pathlib import Path
from PIL import Image

BASE = Path(__file__).resolve().parent.parent
DATA = BASE / "data"

def cache_dir(src_dir, pattern, outdir, size):
    outdir.mkdir(parents=True, exist_ok=True)
    files = sorted(Path(src_dir).glob(pattern))
    done = 0
    for p in files:
        dst = outdir / (p.stem + ".jpg")
        if dst.exists():
            done += 1
            continue
        Image.open(p).convert("RGB").resize(size, Image.LANCZOS).save(dst, quality=90)
        done += 1
    print(f"{size}: {done} cached in {outdir}")

# DENTEX train (705 PNGs, from the training triple JSON)
d = json.load(open(DATA / "training" / "training_data" /
                   "quadrant-enumeration-disease" / "train_quadrant_enumeration_disease.json"))
names = sorted({i["file_name"] for i in d["images"]})
src = DATA / "training" / "training_data" / "quadrant-enumeration-disease" / "xrays"
for size, out in [((1024, 512), DATA / "cache1024" / "dentex_train"),
                 ((512, 256), DATA / "cache512" / "dentex_train")]:
    out.mkdir(parents=True, exist_ok=True)
    done = 0
    for n in names:
        dst = out / (Path(n).stem + ".jpg")
        if dst.exists():
            done += 1
            continue
        Image.open(src / n).convert("RGB").resize(size, Image.LANCZOS).save(dst, quality=90)
        done += 1
    print(f"dentex_train {size}: {done} cached")

# DENTEX val (50 PNGs)
cache_dir(DATA / "validation_data" / "quadrant_enumeration_disease" / "xrays",
          "val_*.png", DATA / "cache1024" / "dentex", (1024, 512))
cache_dir(DATA / "validation_data" / "quadrant_enumeration_disease" / "xrays",
          "val_*.png", DATA / "cache512" / "dentex", (512, 256))
# Tufts (1,000 JPGs)
cache_dir(DATA / "Radiographs", "*.JPG", DATA / "cache1024" / "tufts", (1024, 512))
cache_dir(DATA / "Radiographs", "*.JPG", DATA / "cache512" / "tufts", (512, 256))
print("caches done.")


In [ ]:
# 7b — Build the caches (~5 min).
!python colab/build_caches.py


In [ ]:
# 8 — Label tables (same as the local run): Tufts EDA + harmonisation.
# Regenerates results/tufts_labels.csv, dentex_val_labels.csv and the box CSVs.
!python src/eda_tufts.py && python src/harmonise.py
!ls -la results/*.csv | head


## 9 — Train the full-resolution detector (GPU)

Unfrozen ResNet50-FPN at 1024×512, 5 epochs, mixed precision.
**Expected time:** T4 ~1–2 h, L4 ~40–60 min, A100 ~20–30 min.
A checkpoint is saved after every epoch (`results/fasterrcnn_gpu_ep{N}.pt`).
If the runtime disconnects, resume with:
`!python colab/train_detector_gpu.py --resume results/fasterrcnn_gpu_ep1.pt --start-epoch 2 --lr 0.0005 --epochs 5`
and re-run the Drive-copy cell at the end so far.


In [ ]:
# 9 — Train the GPU detector (long cell — let it run).
!python colab/train_detector_gpu.py --epochs 5


In [ ]:
# 9b — Detector results: compare with the CPU 705 run (mAP 0.27 @0.5).
import json
for f in ("results/detector_metrics_gpu.json", "results/detector_metrics_gpu_thr03.json"):
    print("==", f)
    print(json.dumps(json.load(open(f)), indent=2))


## 10 — Train the classifier on GPU

Same mixed-site ResNet18 recipe as `src/train_classifier_v2.py`
(Tufts + 150 balanced DENTEX-train images, temperature scaling, tuned
threshold). ~5–10 min on a T4.


In [ ]:
# 10 — Train the GPU classifier.
!python colab/train_classifier_gpu.py


In [ ]:
# 10b — Classifier results: compare with v2 (DENTEX val AUC 0.84, Tufts test 0.60).
import json
print(json.dumps(json.load(open("results/classifier_metrics_gpu.json")), indent=2))


## 11 — Save everything to Google Drive

Copies the new metrics, predictions, figures, checkpoints and logs to
`MyDrive/dental-panoramic-ai/colab_run_<date>/`. Run this even if a
training cell failed part-way — the per-epoch checkpoints are what
matter. Then copy the new `*_gpu.json` numbers into
`manuscript/draft.md` by hand (do not paste unreviewed numbers into the
manuscript from here).


In [ ]:
# 11 — Copy all GPU-run outputs to Drive.
import datetime, os, shutil, subprocess
today = datetime.date.today().isoformat()
dest = f"/content/drive/MyDrive/dental-panoramic-ai/colab_run_{today}"
os.makedirs(dest, exist_ok=True)
for pattern in ("*_gpu.json", "*_gpu.csv", "*_gpu.png", "fasterrcnn_gpu_ep*.pt",
                "resnet18_mixed_gpu.pt", "*_gpu_train.log", "detector_val_boxes_gpu.csv"):
    for f in sorted(__import__("glob").glob(f"results/{pattern}")):
        shutil.copy(f, dest)
        print("saved:", os.path.join(dest, os.path.basename(f)))
print("\nAll GPU outputs are in:", dest)
print("Next: copy the new numbers into manuscript/draft.md and update README.md.")
